In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import importlib
import books
import data_io as functions1 
import config 

importlib.reload(books)
importlib.reload(functions1)
importlib.reload(config)

z = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(z)


/Users/alexwebb/laptop_coding/risk_matrix


In [2]:
pf = books.IBKR_live
fetch_csv = functions1.fetch_csv
params = config.params
sort_cols = functions1.sort_cols

GBPUSD = fetch_csv('GBPUSD.FOREX', params=params)
GBPUSD = sort_cols(GBPUSD, ohlc=False)
GBPUSD_last = GBPUSD.iloc[-1]
print('GBPUSD last:', GBPUSD_last)
USD_total = 0
GBP_total = 0
for h in pf:
    exp_usd = h.get("USD_exposure")
    exp_gbp = h.get("GBP_exposure")
    if (exp_usd is None and exp_gbp is None) or h.get('risk_fx') is False:
        print('...skipping', h['name'])
        continue
    print(h['name'],'usd', exp_usd, 'gbp', exp_gbp    )

    # print(h.get("name"),'GBP exp:' ,h.get("GBP_exposure"))
    # get gbpchf last price
    ticker   = h.get("ticker")
    px_df = fetch_csv(f'{ticker}', params=params,)
    px = sort_cols(px_df, ohlc=False).iloc[-1]
    # print('last:' ,px)
    position = h.get("position")
    gpx = .01 if h.get('gbx', True) else 1.0
    mkt_val = position * px * gpx
    print('currency:', h.get('ccy'),'mkt val: £',round(mkt_val))
    if exp_gbp:
        gbp_to_hedge = mkt_val * exp_gbp
        print('gbp to hedge: £',round(gbp_to_hedge))
        GBP_total += gbp_to_hedge

    if exp_usd:
        usd_to_hedge = mkt_val * exp_usd
        if h.get('ccy') == 'GBP':
            print('converting to usd')
            usd_to_hedge *= GBPUSD_last
        USD_total += usd_to_hedge
        print('usd to hedge: $',round(usd_to_hedge))

    print('')
print ('total: £',round(GBP_total))
print ('total: $',round(USD_total))


GBPUSD.FOREX - downloading fresh data
data start date b4 saving: 2019-05-16
GBPUSD last: 1.3124
XMWX usd None gbp 0.13
XMWX.LSE - downloading fresh data
XMWX.LSE - cleaned 2 flat-bar spike(s) on download
data start date b4 saving: 2024-08-27
currency: GBP mkt val: £ 7504
gbp to hedge: £ 976

...skipping EMIM
VUAG usd 1.0 gbp None
VUAG.LSE - downloading fresh data
data start date b4 saving: 2019-05-16
currency: GBP mkt val: £ 2300
converting to usd
usd to hedge: $ 3019

ISJP usd 0 gbp 0.0
ISJP.SW - downloading fresh data
data start date b4 saving: 2019-05-16
currency: JPY mkt val: £ 157640

IEMS usd 0 gbp 0.0
IEMS.LSE - downloading fresh data
data start date b4 saving: 2019-05-16
currency: GBP mkt val: £ 412

NOVN usd 0 gbp 0.0
NOVN.SW - downloading fresh data
data start date b4 saving: 2019-05-16
currency: CHF mkt val: £ 1989

...skipping CASH_JPY
total: £ 976
total: $ 3019


In [ ]:
import json

# Cache metadata summary for tickers touched in this notebook run.
watch = {"GBPUSD.FOREX"}
for h in pf:
    t = h.get("ticker")
    if h.get("risk_fx") is not False and t:
        watch.add(t)

print("CACHE META SUMMARY")
for t in sorted(watch):
    meta_path = functions1.cache_meta_path(t)
    if not meta_path.exists():
        print(t, "-> MISSING")
        continue

    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    mode = meta.get("last_update_mode")
    last_update = meta.get("last_update_utc")
    last_full = meta.get("last_full_refresh_utc")
    print(t, "->", mode, "|", "last_update:", last_update, "|", "last_full:", last_full)